In [1]:
import pandas as pd
from migration_utils import migrate_financial_statements, traded_companies_fields, migrate_traded_companies, balance_sheet_fields, get_connection
import psycopg2, os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import re
import numpy as np
import json
from sqlalchemy import create_engine

In [5]:
file_name = "Data/canada/bulk_annual_financials_canada.csv"

In [10]:
data = pd.read_csv(file_name,na_values=[], keep_default_na=False)

/var/folders/0k/p7xpfjb15zq_sj5w2mwmzv8w0000gn/T/ipykernel_22488/1916032985.py:1: DtypeWarning: Columns (9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,35,36,50,52,53,54,55,56,57,58,59,65,66,67,69,70,71,72,73,75,76,77,87,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,137,138,141,142,143,144,145,146,147,148,149,151,152,153,154,155,156,176,177,178,182,183,184,185,187,189,190,191,194,195,196,197,198,199,201,202,203,204,205,209,210,211,212,213,214,215) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_name,na_values=[], keep_default_na=False)


In [7]:
data.head()

,symbol,qfs_symbol,exchange,name,company_type,currency,industry,period_type,period_end_date,revenue,...,assets_to_equity_median,debt_to_assets_median,debt_to_equity_median,roi_median,equity_to_assets_median,underwriting_margin_median,nim_median,earning_assets_to_equity_median,loans_to_deposits_median,loan_loss_reserve_to_loans_median
0,ALBA.P,ALBA.P:CA,TSXVenture,A-Labs Capital V Corp.,normal,CAD,Diversified Financial Services,FY,2019-12,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ALBA.P,ALBA.P:CA,TSXVenture,A-Labs Capital V Corp.,normal,CAD,Diversified Financial Services,FY,2020-12,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ALBA.P,ALBA.P:CA,TSXVenture,A-Labs Capital V Corp.,normal,CAD,Diversified Financial Services,FY,2021-12,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ALBA.P,ALBA.P:CA,TSXVenture,A-Labs Capital V Corp.,normal,CAD,Diversified Financial Services,FY,TTM,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,EBR,EBR:CA,CSE,Eagle Bay Resources Corp.,normal,CAD,Metals & Mining,FY,2022-07,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
filtered_df = data[data['name'] == 'National Bank of Canada']

In [12]:
filtered_df

,symbol,qfs_symbol,exchange,name,company_type,currency,industry,period_type,period_end_date,revenue,...,assets_to_equity_median,debt_to_assets_median,debt_to_equity_median,roi_median,equity_to_assets_median,underwriting_margin_median,nim_median,earning_assets_to_equity_median,loans_to_deposits_median,loan_loss_reserve_to_loans_median
74969,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2001-10,3332000000.0,...,NA,,NA,,NA,,NA,NA,NA,NA
74970,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2002-10,3518000000.0,...,19.74666705340499,,0.4041231298246768,,0.05072160508365668,,0.02061149519619064,18.757766318631056,0.8980733240893495,0.020090385979322292
74971,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2003-10,3539000000.0,...,18.802572745537823,,0.3937323065872312,,0.053188021470069013,,0.02061149519619064,17.788196904654868,0.8958911766460671,0.019684131887028285
74972,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2004-10,3635000000.0,...,19.02951972828585,,0.37867284474677476,,0.05257157447999662,,0.019795065306761762,17.202226209320898,0.8890148183603527,0.018429978091777433
74973,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2005-10,3736000000.0,...,19.02951972828585,,0.36423420367769893,,0.05257157447999662,,0.019310684689210968,17.115248650261808,0.8890148183603527,0.017348495888276444
74974,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2006-10,3795000000.0,...,19.02951972828585,,0.36423420367769893,,0.05257157447999662,,0.019144552991212777,17.115248650261808,0.8575166997886343,0.016702680071940774
74975,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2007-10,3417000000.0,...,19.02951972828585,,0.35698984923122856,,0.05257157447999662,,0.018690310275195618,16.76438057541465,0.8186256556075029,0.015312281997823037
74976,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2008-10,3542000000.0,...,18.490012852599754,,0.362782494467369,,0.05408697427129255,,0.018690310275195618,16.46221569326751,0.7950826425263002,0.014196964843520862
74977,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2009-10,4131000000.0,...,17.993600716086362,,0.3850863107542958,,0.05559551480859855,,0.018600362247510344,16.297797079041743,0.7516710576176451,0.012871930745462999
74978,NA,NA:CA,Toronto,National Bank of Canada,bank,CAD,Banks,FY,2010-10,4289000000.0,...,18.87610309370612,,0.4041231298246768,,0.05302034872550157,,0.018257334831416398,16.240957846503683,0.7516710576176451,0.011297349492067663


In [4]:
#remove rows where ticker is nan
data = data[data['symbol'].notna()]



In [5]:
#create sql alchemy connection
engine = get_connection()

#create psycopg2 connection
psy_connection = psycopg2.connect(host="localhost", database="value-investing-dev", user="postgres", password="v,1846PSVv,1846PSV")


In [6]:
migr_trad_comp = migrate_traded_companies(sqlalchemy_engine=engine, quickfs_df=data, psycopg2_connection=psy_connection, relevant_fields=traded_companies_fields, target_table='quickfs_tradedcompanies')

migr_balance_quart = migrate_financial_statements(sqlalchemy_engine=engine, quickfs_df=data, psycopg2_connection=psy_connection, relevant_fields=balance_sheet_fields, target_table='quickfs_balancesheetquarter')

migration quickfs_tradedcompanies, file:  started!
migration quickfs_tradedcompanies, file:  ended! Nr. rows migrated: 0


/Users/pa/Desktop/SideProjects/Valuation/value-investing-app/value-investing-backend/migrations/quickfs_database/migration_utils.py:97: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_extracted.rename(columns=rename_columns, inplace=True)


migration table quickfs_balancesheetquarter, file:  started!
migration table quickfs_balancesheetquarter, file:  end! Nr. rows migrated: 0


Exception: Nr. of migrated rows: 250040 is not the same as new rows in database: 500055, for file: 

In [7]:
target_table = 'quickfs_balancesheetquarter'

#select elements from database
df_extracted = data[balance_sheet_fields]

#rename columns
df_extracted.rename(columns={'symbol' : 'ticker_id'}, inplace=True)

#create string to extract relevant coluns
cols = ', '.join(list(df_extracted.columns))

#define select query to extract existing rows in database
query = f"SELECT {cols} from public.{target_table}"
try:
    #create a psycopg cursor to execute query
    cursor = psy_connection.cursor()

    #execute query
    cursor.execute(query)

    #transform database query content into dataframe
    df_database = pd.DataFrame(cursor.fetchall(), columns=list(df_extracted.columns))
except Exception as e:
    print('exception: ', e)
    cursor.close()

/var/folders/0k/p7xpfjb15zq_sj5w2mwmzv8w0000gn/T/ipykernel_4380/238040605.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_extracted.rename(columns={'symbol' : 'ticker_id'}, inplace=True)


In [9]:
def transform_date_to_string(date):
    return date.strftime('%Y-%m-%d')

def transform_date(date):
    return date + "-01"

In [10]:
#transform date column from quickfs format YYYY-MM to django compatible format YYYY-MM-DD
df_extracted['period_end_date'] = df_extracted['period_end_date'].apply(transform_date)


/var/folders/0k/p7xpfjb15zq_sj5w2mwmzv8w0000gn/T/ipykernel_89872/1672763532.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_extracted['period_end_date'] = df_extracted['period_end_date'].apply(transform_date)


In [11]:
df_database['period_end_date'] = df_database['period_end_date'].apply(transform_date_to_string)


In [22]:
fields = ["symbol", "period_end_date", "cash_and_equiv", "st_investments", "receivables", "inventories", "other_current_assets", "total_current_assets", "equity_and_other_investments", "ppe_gross", "accumulated_depreciation", "ppe_net", "intangible_assets", "goodwill", "other_lt_assets", "total_assets", "accounts_payable", "tax_payable", "current_accrued_liabilities", "st_debt", "current_deferred_revenue", "current_deferred_tax_liability", "current_capital_leases", "other_current_liabilities", "total_current_liabilities", "lt_debt", "noncurrent_capital_leases", "pension_liabilities", "noncurrent_deferred_revenue", "other_lt_liabilities", "total_liabilities", "common_stock", "preferred_stock", "retained_earnings", "aoci", "apic", "treasury_stock", "other_equity", "minority_interest_liability", "total_equity", "total_liabilities_and_equity", "total_investments", "deferred_policy_acquisition_cost", "unearned_premiums", "future_policy_benefits", "loans_gross", "allowance_for_loan_losses", "unearned_income", "loans_net", "deposits_liability"]

#replace symbol with ticker_id
list(df_extracted.columns)

['ticker_id',
 'period_end_date',
 'cash_and_equiv',
 'st_investments',
 'receivables',
 'inventories',
 'other_current_assets',
 'total_current_assets',
 'equity_and_other_investments',
 'ppe_gross',
 'accumulated_depreciation',
 'ppe_net',
 'intangible_assets',
 'goodwill',
 'other_lt_assets',
 'total_assets',
 'accounts_payable',
 'tax_payable',
 'current_accrued_liabilities',
 'st_debt',
 'current_deferred_revenue',
 'current_deferred_tax_liability',
 'current_capital_leases',
 'other_current_liabilities',
 'total_current_liabilities',
 'lt_debt',
 'noncurrent_capital_leases',
 'pension_liabilities',
 'noncurrent_deferred_revenue',
 'other_lt_liabilities',
 'total_liabilities',
 'common_stock',
 'preferred_stock',
 'retained_earnings',
 'aoci',
 'apic',
 'treasury_stock',
 'other_equity',
 'minority_interest_liability',
 'total_equity',
 'total_liabilities_and_equity',
 'total_investments',
 'deferred_policy_acquisition_cost',
 'unearned_premiums',
 'future_policy_benefits',
 'loan

In [23]:
all_df = pd.merge(df_extracted, df_database, on=list(df_extracted.columns), how='left', indicator='exists')

In [24]:
all_df['exists'] = np.where(all_df.exists == 'both', True, False)


In [25]:
all_df[all_df['exists'] == False].shape

(0, 51)

In [14]:
all_df.head(100)

,ticker_id,period_end_date,cash_and_equiv,st_investments,receivables,inventories,other_current_assets,total_current_assets,equity_and_other_investments,ppe_gross,...,total_investments,deferred_policy_acquisition_cost,unearned_premiums,future_policy_benefits,loans_gross,allowance_for_loan_losses,unearned_income,loans_net,deposits_liability,exists
0,ANNX,2021-09-01,68519000.0,202848000.0,231000.0,0.0,3900000.0,275498000.0,0.0,36506000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
1,ANNX,2020-03-01,33348000.0,0.0,6000.0,0.0,1155000.0,34509000.0,0.0,3442000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
2,ANNX,2019-12-01,43931000.0,0.0,79000.0,0.0,1396000.0,45406000.0,0.0,3442000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
3,ANNX,2019-06-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
4,ANNX,2019-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,MYNDF,2022-04-01,10000.0,0.0,95000.0,0.0,556000.0,661000.0,0.0,135000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
96,MYNDF,2023-01-01,0.0,0.0,107000.0,0.0,1000.0,108000.0,0.0,90000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
97,MYNDF,2022-07-01,0.0,0.0,103000.0,0.0,313000.0,416000.0,0.0,124000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
98,MYNDF,2022-01-01,127000.0,0.0,79000.0,0.0,794000.0,1000000.0,0.0,264000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both


In [ ]:
diff